In [11]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [12]:
train_data= pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

### Data- Preprocessing

In [13]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text)
    text = re.sub(r"\s+"," ",text)
    text = re.sub(r"<.*?>"," ",text)
    text.strip().lower()
    return text 

In [15]:
train_data["dialogue"] = train_data["dialogue"].fillna("").astype(str).apply(clean_data)
train_data["summary"] = train_data["summary"].fillna("").astype(str).apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].fillna("").astype(str).apply(clean_data)
val_data["summary"] = val_data["summary"].fillna("").astype(str).apply(clean_data)

In [16]:
train_data["dialogue"][0]

"Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)"

### Tokenize

In [17]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json: 0.00B [00:00, ?B/s]

d:\Vs code\AIML\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abhay Kumar\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
# raw data =>tokenize 

def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets = tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [21]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()